In [ ]:
!pip install requests
!pip install spotipy --quiet

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from google.colab import userdata, drive
import os

# Link with spotify API app
CLIENT_ID = userdata.get('Client_ID')
CLIENT_SECRET = userdata.get('Client_Secret')

# Spotify API callback
REDIRECT_URI = 'http://127.0.0.1:8888/callback'

# Scopes determine what you can do, Spotify API
SCOPE = 'user-library-read playlist-modify-public playlist-read-private'

# Store access tokens for a specific user in google drive
# So you don't have to log in over and over while testing
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    # Save the cache file in Drive root
    cache_path = '/content/drive/MyDrive/.spotify_cache'
else:
    cache_path = '.spotify_cache'

# THE AUTHENTICATION
sp_oauth = SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri=REDIRECT_URI,
    scope=SCOPE,
    cache_path=cache_path,
    show_dialog=True
)

# Check if we already have a valid token in cache
token_info = sp_oauth.get_cached_token()

if not token_info:
    # If not, generate the authorization URL
    auth_url = sp_oauth.get_authorize_url()
    print(f"\nClick this link to authorize: \n{auth_url}")
    print("\n--- INSTRUCTIONS ---")
    print("1. Click the link above and sign in.")
    print("2. You will see a 'This site can't be reached' error page (localhost).")
    print("3. COPY the entire URL from your browser's address bar.")
    print("4. PASTE it into the box below.\n")

    # Wait for user input
    response_url = input("Paste the redirected URL here: ")

    # Parse the code from the URL and swap for token
    code = sp_oauth.parse_response_code(response_url)
    token_info = sp_oauth.get_access_token(code)

# Spotify Client
sp = spotipy.Spotify(auth=token_info['access_token'])

print(f"\nSuccess! Logged in as: {sp.me()['display_name']}")
print(f"Token expires at: {token_info['expires_at']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Success! Logged in as: lalithok
Token expires at: 1764807004


In [ ]:
def find_playlist_id(playlist_name):
    """
    Searches the logged-in user's playlists for a specific name.
    Returns the ID if found, None if not.
    """
    print(f"Searching for playlist: '{playlist_name}'...")

    # Get the first batch of playlists
    playlists = sp.current_user_playlists(limit=50)

    while playlists:
        for playlist in playlists['items']:
            if playlist['name'].lower() == playlist_name.lower():
                return playlist['id']

        # Check if there are more playlists to fetch
        if playlists['next']:
            playlists = sp.next(playlists)
        else:
            playlists = None

    return None

def get_all_tracks_from_playlist(playlist_id):
    # Fetches EVERY track from a playlist, handling pagination automatically.
    results = sp.playlist_items(playlist_id)
    tracks = results['items']

    # Loop to get the rest of the tracks (if more than 100)
    while results['next']:
        results = sp.next(results)
        tracks.extend(results['items'])

    return tracks

In [ ]:
!pip install transformers torch librosa datasets --quiet
import requests
import torch
import os
import librosa
import numpy as np
from transformers import ClapModel, ClapProcessor

In [ ]:
# DEEZER DOWNLOADER - Gets clips of songs

def download_preview(track_name, artist_name, output_filename="temp.mp3"):
    base_url = "https://api.deezer.com/search"
    params = {'q': f'artist:"{artist_name}" track:"{track_name}"', 'limit': 1}

    try:
        if os.path.exists(output_filename):
            os.remove(output_filename)

        response = requests.get(base_url, params=params)
        data = response.json()

        if 'data' not in data or not data['data']:
            return None # Not found

        preview_url = data['data'][0].get('preview')
        if not preview_url:
            return None

        # Download
        audio_response = requests.get(preview_url)
        with open(output_filename, 'wb') as f:
            f.write(audio_response.content)
        return output_filename

    except Exception as e:
        print(f"Error downloading {track_name}: {e}")
        return None

In [ ]:
# Transformer SETUP - CLAP Model from HuggingFace

print("\nLoading CLAP Model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = ClapModel.from_pretrained("laion/larger_clap_music").to(device)
processor = ClapProcessor.from_pretrained("laion/larger_clap_music")
print(f"Model loaded on {device.upper()}")

def get_embedding(audio_path):
    # Load audio at 48k (required by CLAP)
    audio_data, _ = librosa.load(audio_path, sr=48000)

    # Crop to 30s max (Deezer previews are 30s anyway)
    audio_data = audio_data[:48000 * 30]

    inputs = processor(audios=audio_data, return_tensors="pt", sampling_rate=48000).to(device)

    with torch.no_grad():
        outputs = model.get_audio_features(**inputs)

    # Move back to CPU and convert to numpy for storage
    return outputs.cpu().numpy()


Loading CLAP Model...
Model loaded on CUDA


In [ ]:
# Embeddings for Playlist tracks

target_playlist = input("\nEnter the name of your playlist: ")
pid = find_playlist_id(target_playlist)

if pid:
    tracks = get_all_tracks_from_playlist(pid)
    print(f"Found {len(tracks)} tracks. Starting processing...\n")

    dataset = []

    for i, item in enumerate(tracks):
        if not item['track']: continue

        name = item['track']['name']
        artist = item['track']['artists'][0]['name']
        print(f"[{i+1}/{len(tracks)}] Processing: {name} - {artist}")

        # Download
        file_path = download_preview(name, artist)

        if file_path:
            # Embed
            try:
                vec = get_embedding(file_path)

                # Store Data
                dataset.append({
                    "name": name,
                    "artist": artist,
                    "embedding": vec,
                    "spotify_id": item['track']['id']
                })
            except Exception as e:
                print(f"---- Embedding failed: {e}")

            # Cleanup temp file
            if os.path.exists(file_path):
                os.remove(file_path)
        else:
            print("---- Audio not found on Deezer.")

    print(f"\nDone! Generated embeddings for {len(dataset)} songs.")

else:
    print("---- Playlist not found.")


Enter the name of your playlist: jigsaw
Searching for playlist: 'jigsaw'...
Found 42 tracks. Starting processing...

[1/42] Processing: Vogel Im Kafig - Sawano Hiroyuki


/tmp/ipython-input-2180363023.py:16: FutureWarning: `audios` is deprecated and will be removed in version v4.59.0 for `ClapProcessor.__call__`. Use `audio` instead.
  inputs = processor(audios=audio_data, return_tensors="pt", sampling_rate=48000).to(device)


[2/42] Processing: life force - ptasinski
[3/42] Processing: In This Shirt - The Irrepressibles
[4/42] Processing: As the world caves in (Outro) - Semi
[5/42] Processing: Idea 10 - Gibran Alcocer
[6/42] Processing: If I Am With You - 照井順政
[7/42] Processing: Solas - Jamie Duffy
[8/42] Processing: Ethereal - Txmy
[9/42] Processing: Swan Serenade - Piano House
[10/42] Processing: I. SHINJI - Shiro SAGISU
[11/42] Processing: The Eternal Bond of Stargazing - Murr
[12/42] Processing: Idea 15 - Gibran Alcocer
[13/42] Processing: Ballerina - Yehezkel Raz
[14/42] Processing: Lobster - RJ Pasin
[15/42] Processing: Van Gogh - Virginio Aiello
[16/42] Processing: Idea 22 - Gibran Alcocer
[17/42] Processing: Cornfield Chase - Hans Zimmer
[18/42] Processing: No Time for Caution - Hans Zimmer
[19/42] Processing: Time - Hans Zimmer
[20/42] Processing: Experience - Ludovico Einaudi
[21/42] Processing: Revelations - Eternal Eclipse
[22/42] Processing: Shape of Lies - Eternal Eclipse
[23/42] Processing: L

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

In [ ]:
if len(dataset) > 2:
    print("\n--- CLUSTERING ---")

    # Prepare Matrix
    raw_X = np.array([d['embedding'].flatten() for d in dataset])

    # --- SANITY CHECK: Is the data distinct? ---
    # If this prints 0.0, your embeddings are all identical!
    variance = np.var(raw_X)
    print(f" Data Variance: {variance:.6f}")
    if variance < 0.000001:
        print(" |||||||| CRITICAL WARNING: Your embeddings are all the same (or empty)!")
        print("   The clustering will not work until the data extraction is fixed.")

    # Apply Transformations
    X_norm = normalize(raw_X)

    # PCA Reduction
    n_components = min(20, len(dataset))
    pca = PCA(n_components=n_components)
    X_final = pca.fit_transform(X_norm)

    # --- SANITY CHECK: Did the shape change? ---
    print(f" Original Shape: {raw_X.shape}")
    print(f" New Shape:      {X_final.shape}")

    if raw_X.shape == X_final.shape:
        print("---- WARNING: PCA was not applied. The shapes are the same.")

    # Clustering
    k_input = input(f"\nEnter number of clusters (2-{len(dataset)-1}) or press ENTER: ")

    optimal_k = 0

    if k_input.strip() == "":
        print(" Auto-detecting optimal K...")
        max_k = min(10, len(dataset))
        best_score = -1

        for k in range(2, max_k):
            kmeans = KMeans(n_clusters=k, random_state=None, n_init=10)
            labels = kmeans.fit_predict(X_final)
            score = silhouette_score(X_final, labels)
            print(f"   K={k}: Score={score:.4f}")

            if score > best_score:
                best_score = score
                optimal_k = k
    else:
        optimal_k = int(k_input)

    print(f" Using K: {optimal_k}")

    # Final Fit
    kmeans = KMeans(n_clusters=optimal_k, random_state=None, n_init=10)
    cluster_labels = kmeans.fit_predict(X_final)

    # Print Results
    clusters = {i: [] for i in range(optimal_k)}
    for idx, label in enumerate(cluster_labels):
        clusters[label].append(dataset[idx])

    for cluster_id, songs in clusters.items():
        print(f"\n📂 Cluster {cluster_id + 1}:")
        for s in songs[:5]:
            print(f"   - {s['name']} by {s['artist']}")


--- CLUSTERING ---
 Data Variance: 0.001948
 Original Shape: (42, 512)
 New Shape:      (42, 20)

Enter number of clusters (2-41) or press ENTER: 
 Auto-detecting optimal K...
   K=2: Score=0.2282
   K=3: Score=0.2294
   K=4: Score=0.2507
   K=5: Score=0.2156
   K=6: Score=0.2083
   K=7: Score=0.1735
   K=8: Score=0.1618
   K=9: Score=0.1497
 Using K: 4

📂 Cluster 1:
   - In This Shirt by The Irrepressibles
   - If I Am With You by 照井順政
   - Ethereal by Txmy
   - The Eternal Bond of Stargazing by Murr
   - Cornfield Chase by Hans Zimmer

📂 Cluster 2:
   - Vogel Im Kafig by Sawano Hiroyuki
   - As the world caves in (Outro) by Semi
   - Idea 10 by Gibran Alcocer
   - Idea 15 by Gibran Alcocer
   - Van Gogh by Virginio Aiello

📂 Cluster 3:
   - a good man with a broken heart by LoVibe.
   - Fairytale (Violin) by Dramatic Violin

📂 Cluster 4:
   - life force by ptasinski
   - Solas by Jamie Duffy
   - Swan Serenade by Piano House
   - I. SHINJI by Shiro SAGISU
   - Ballerina by Yehezkel 

In [ ]:
"""import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Get the embeddings
X = np.array([d['embedding'].flatten() for d in dataset])

# 2. Calculate Similarity Matrix (0.0 to 1.0)
# 1.0 means identical, 0.0 means completely opposite
sim_matrix = cosine_similarity(X)

print("\n--- 🔍 SIMILARITY ANALYSIS ---")
print("How similar is each song to the others? (0% to 100%)\n")

for i in range(len(dataset)):
    # Get the song name
    name_i = dataset[i]['name']

    # Find the most similar OTHER song
    # We set index [i] to -1 so it doesn't pick itself
    scores = sim_matrix[i].copy()
    scores[i] = -1

    best_match_idx = np.argmax(scores)
    best_score = scores[best_match_idx]

    match_name = dataset[best_match_idx]['name']

    print(f"🎵 '{name_i}'")
    print(f"   ↳ Most similar to: '{match_name}' ({best_score:.1%} match)")
    print("-" * 40)
"""


'import numpy as np\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n# 1. Get the embeddings\nX = np.array([d[\'embedding\'].flatten() for d in dataset])\n\n# 2. Calculate Similarity Matrix (0.0 to 1.0)\n# 1.0 means identical, 0.0 means completely opposite\nsim_matrix = cosine_similarity(X)\n\nprint("\n--- 🔍 SIMILARITY ANALYSIS ---")\nprint("How similar is each song to the others? (0% to 100%)\n")\n\nfor i in range(len(dataset)):\n    # Get the song name\n    name_i = dataset[i][\'name\']\n\n    # Find the most similar OTHER song\n    # We set index [i] to -1 so it doesn\'t pick itself\n    scores = sim_matrix[i].copy()\n    scores[i] = -1\n\n    best_match_idx = np.argmax(scores)\n    best_score = scores[best_match_idx]\n\n    match_name = dataset[best_match_idx][\'name\']\n\n    print(f"🎵 \'{name_i}\'")\n    print(f"   ↳ Most similar to: \'{match_name}\' ({best_score:.1%} match)")\n    print("-" * 40)\n'

In [ ]:
vibe_library = {
    # --- POP & ELECTRONIC ---
    "Pop": "Upbeat radio music with catchy melodies, female or male vocals, and polished production",
    "Synth-Pop": "80s style synthesizers, retro electronic drums, new wave vibe, catchy beat",
    "Hyperpop": "Distorted bass, pitched up vocals, glitchy electronic textures, very fast tempo",
    "Indie Pop": "Jangly electric guitars, soft vocals, catchy melody, bedroom studio sound",
    "EDM": "High energy electronic dance music, heavy bass drop, synthesizer lead, club atmosphere",
    "House": "Four-on-the-floor drum beat, 120 bpm, groovy bassline, soulful vocals, dance floor",
    "Techno": "Repetitive mechanical rhythm, synthesized textures, minimal melody, dark atmosphere",
    "Lo-Fi": "Low fidelity hip hop beat, vinyl crackle noise, slow tempo, relaxing piano loop",

    # --- ROCK & ALTERNATIVE ---
    "Rock": "Distorted electric guitars, heavy drums, energetic male vocals, classic rock band sound",
    "Indie Rock": "Alternative rock band, raw guitar sound, garage recording feel, energetic but unpolished",
    "Punk": "Fast tempo, aggressive power chords, shouting vocals, high energy, raw production",
    "Psychedelic": "Reverb-heavy guitars, delay effects, trippy atmosphere, 60s rock vibe",
    "Soft Rock": "Acoustic guitars, light drums, smooth vocals, relaxing rock ballad",
    "Metal": "Heavily distorted down-tuned guitars, aggressive drumming, double kick pedal, screaming vocals",

    # --- HIP HOP & R&B ---
    "Hip Hop": "Boom bap drum beat, rapping male vocals, sampling culture, rhythmic flow",
    "Trap": "808 sub bass, rapid hi-hat rolls, autotuned vocals, modern rap production",
    "R&B": "Smooth soulful vocals, slow groove, electronic beats, romantic atmosphere",
    "Neo-Soul": "Jazzy chords, laid back hip hop beat, soulful vocals, organic instrumentation",

    # --- MOODS (Describing the Sonic Texture) ---
    "Happy": "Major key, uptempo, bright instrumentation, clapping, joyful atmosphere",
    "Sad": "Minor key, slow tempo, melancholic piano or strings, emotional vocals",
    "Aggressive": "Loud volume, distorted textures, fast tempo, shouting, chaotic energy",
    "Chill": "Slow tempo, soft dynamics, minimal instrumentation, relaxing atmosphere",
    "Dark": "Low frequencies, ominous drone, dissonant chords, creepy atmosphere, minor key",
    "Euphoric": "Uplifting chord progression, large soundstage, crescendo, emotional high",
    "Romantic": "Slow tempo, strings section, soft piano, intimate vocals, love song vibe",

    # --- CONTEXTS (Describing the Environment) ---
    "Workout": "High bpm, consistent rhythm, heavy bass, motivating energy, gym music",
    "Party": "Loud, upbeat, danceable rhythm, celebratory vibe, crowd noise textures",
    "Focus": "Instrumental only, no vocals, repetitive calm pattern, background music",
    "Night Drive": "Synthwave aesthetic, steady beat, atmospheric pads, neon city vibe",
    "Summer": "Tropical percussion, bright acoustic guitar, upbeat, sunny atmosphere",
    "Rainy Day": "Soft acoustic instruments, sound of rain, melancholic jazz, cozy vibe"
}

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

display_labels = list(vibe_library.keys())
descriptive_prompts = list(vibe_library.values())

print(f"\n  Preparing {len(display_labels)} Descriptive Embeddings...")

# EMBED THE DESCRIPTIONS

# We embed the descriptions (the long text), not the labels
text_inputs = processor(text=descriptive_prompts, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    label_features = model.get_text_features(**text_inputs)

# Normalize labels
label_matrix = label_features.cpu().numpy()
label_matrix = normalize(label_matrix)

# Dictionary to store the final combined name for each cluster
cluster_names = {}

# THE WEIGHTED ELECTION

print("---  CASTING VOTES (DESCRIPTIVE MATCHING) ---")

for c_id, songs in clusters.items():
    if not songs:
        continue

    # Get Audio Embeddings for this Cluster
    song_matrix = np.array([s['embedding'].flatten() for s in songs])
    song_matrix = normalize(song_matrix)

    # Calculate Similarity Matrix (Audio vs Descriptions)
    sim_matrix = cosine_similarity(song_matrix, label_matrix)

    # Weighted Voting Logic
    cluster_vote_tally = defaultdict(float)

    for i in range(len(songs)):
        scores = sim_matrix[i]

        # Get Top 5 Best Fits
        top_5_indices = np.argsort(scores)[-5:][::-1]

        # Cast Votes
        for idx in top_5_indices:
            score = scores[idx]

            # IGNORE NEGATIVE MATCHES
            if score > 0:
                # Match the index back to the DISPLAY LABEL
                # We matched on the description, but we vote for the label
                label_name = display_labels[idx]
                cluster_vote_tally[label_name] += score

    # Determine the Winners & Standardize
    final_scores = []
    for label, total_impact in cluster_vote_tally.items():
        std_score = total_impact / len(songs)
        final_scores.append((label, std_score))

    # Sort and Name
    sorted_votes = sorted(final_scores, key=lambda x: x[1], reverse=True)

    if not sorted_votes:
        final_name = "Eclectic Mix"
    else:
        top_3_labels = [item[0] for item in sorted_votes[:3]]
        final_name = " + ".join(top_3_labels)

    cluster_names[c_id] = final_name

    # --- PRINT RESULTS ---
    print(f"\n📂 Cluster {c_id + 1}: {final_name}")
    print(f"   ℹ  Cluster Size: {len(songs)} songs")
    print(f"     Top Matches (Standardized):")

    for label, std_score in sorted_votes[:3]:
        bar_len = int(std_score * 20)
        bar = ""
        print(f"      - {label:<15} {bar} ({std_score:.3f})")

    print(f"     Example: {songs[0]['name']}")


  Preparing 31 Descriptive Embeddings...
---  CASTING VOTES (DESCRIPTIVE MATCHING) ---

📂 Cluster 1: Focus + Chill + Lo-Fi
   ℹ  Cluster Size: 6 songs
     Top Matches (Standardized):
      - Focus            (0.001)
      - Chill            (0.001)
      - Lo-Fi            (0.001)
     Example: In This Shirt

📂 Cluster 2: Focus + Chill + Lo-Fi
   ℹ  Cluster Size: 17 songs
     Top Matches (Standardized):
      - Focus            (0.002)
      - Chill            (0.002)
      - Lo-Fi            (0.002)
     Example: Vogel Im Kafig

📂 Cluster 3: Eclectic Mix
   ℹ  Cluster Size: 2 songs
     Top Matches (Standardized):
     Example: a good man with a broken heart

📂 Cluster 4: Focus + Chill + Lo-Fi
   ℹ  Cluster Size: 17 songs
     Top Matches (Standardized):
      - Focus            (0.008)
      - Chill            (0.008)
      - Lo-Fi            (0.007)
     Example: life force


In [ ]:
user_id = sp.current_user()['id']
print(f"  Creating playlists for User ID: {user_id}")

# Create Playlists from Clusters
for cluster_id, songs in clusters.items():
    if not songs:
        continue # Skip empty clusters

    # Define Playlist Details
    voted_name = cluster_names.get(cluster_id, "Unknown Vibes")

    description = f"AI Cluster: {voted_name}. Generated by voting on {len(songs)} songs."

    # Create the Empty Playlist on Spotify
    new_playlist = sp.user_playlist_create(
        user=user_id,
        name=f"{voted_name} (AI Mix)", # Result: "Pop + Happy + Summer "
        public=True,
        description=f"An AI generated mix focusing on {voted_name}."
    )

    new_playlist_id = new_playlist['id']
    print(f"\n Created Playlist: '{voted_name}' (ID: {new_playlist_id})")

    # Prepare Track URIs
    # Spotify API expects URIs in the format: "spotify:track:YOUR_ID"
    track_uris = []
    for s in songs:
        if 'spotify_id' in s and s['spotify_id']:
            track_uris.append(f"spotify:track:{s['spotify_id']}")
        else:
            print(f"---- Skipping '{s['name']}' - Missing Spotify ID.")

    # Add Tracks (Batching for large lists)
    if track_uris:
        batch_size = 100
        for i in range(0, len(track_uris), batch_size):
            batch = track_uris[i : i + batch_size]
            sp.playlist_add_items(new_playlist_id, batch)
            print(f"   ➡️ Added {len(batch)} tracks to the playlist.")
    else:
        print("---- No valid tracks found to add.")

print("\n  Process Complete!")

  Creating playlists for User ID: lalithok

 Created Playlist: 'Focus + Chill + Lo-Fi' (ID: 42846bEKZuTqRWWzCsQup8)
   ➡️ Added 6 tracks to the playlist.

 Created Playlist: 'Focus + Chill + Lo-Fi' (ID: 7CMBWRBBbSaX8niM1vzSpA)
   ➡️ Added 17 tracks to the playlist.

 Created Playlist: 'Eclectic Mix' (ID: 0qETE3Dkx4UMQB3if7oLeU)
   ➡️ Added 2 tracks to the playlist.

 Created Playlist: 'Focus + Chill + Lo-Fi' (ID: 6CkLlg8Zvew7iWxs0MVXNg)
   ➡️ Added 17 tracks to the playlist.

  Process Complete!
